# Решения: memo + DP 1D

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import csv


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


def load_coin_cases() -> list[dict[str, object]]:
    path = _find('coin_change_cases.csv')
    rows: list[dict[str, object]] = []
    with path.open(encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append({
                'case_id': row['case_id'],
                'amount': int(row['amount']),
                'coins': [int(x) for x in row['coins'].split()],
                'expected_min_coins': int(row['expected_min_coins']),
            })
    return rows


def load_grid() -> list[list[int]]:
    path = _find('route_cost_grid_4x5.csv')
    grid: list[list[int]] = []
    with path.open(encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            grid.append([int(x) for x in row])
    return grid


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def ways(n: int) -> int:
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)

def min_coins(amount: int, coins: list[int]) -> int:
    inf = amount + 1
    dp = [inf] * (amount + 1)
    dp[0] = 0
    for s in range(1, amount + 1):
        best = inf
        for c in coins:
            if s - c >= 0 and dp[s - c] + 1 < best:
                best = dp[s - c] + 1
        dp[s] = best
    return -1 if dp[amount] == inf else dp[amount]

def max_safe_gain(points: list[int]) -> int:
    if not points:
        return 0
    take, skip = 0, 0
    for value in points:
        new_take = skip + value
        skip = max(skip, take)
        take = new_take
    return max(take, skip)

cases = load_coin_cases()
for row in cases:
    got = min_coins(row['amount'], row['coins'])
    assert got == row['expected_min_coins']

assert ways(10) == 89
route = [8, 4, 5, 9, 3, 1, 7]
assert max_safe_gain(route) == 24
assert min_coins(27, [1, 5, 10]) == 5
assert min_coins(6, [4, 7]) == -1
MEMO_NOTE = (
    'Мемоизация убирает повторный пересчёт одинаковых состояний рекурсии. '
    'В инженерии это снижает время ответа, когда подзадачи часто повторяются, как в размене и шагах по маршруту.'
)
print('ways(10)=', ways(10))
print('coin cases OK')
print('max_safe_gain=', max_safe_gain(route))
print(MEMO_NOTE)